# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Croissant dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Explore the dataset metadata
md = dataset.metadata
print(f"Dataset: {md.name}\n")
print(md.description)

# Optional: Show key metadata fields
print(f"\nPublished: {getattr(md, 'datePublished', 'Unknown')}")
print(f"Version: {getattr(md, 'version', 'Unknown')}")
print(f"Identifier: {getattr(md, 'identifier', 'Unknown')}")

## 2. Data Overview
Review the structure of available record sets and their fields using their `@id` values.

In [ ]:
# List the available record sets and the field IDs for each (using @id)
if hasattr(dataset, 'record_sets'):
    print("Available record sets:")
    for record_set in dataset.record_sets:
        print(f"- {record_set['@id']} (name: {record_set.get('name', '-')})")
        if 'field' in record_set:
            fields = record_set['field']
            if not isinstance(fields, list):
                fields = [fields]
            for field in fields:
                # field can be a dict or a str
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
                print(f"    - field @id: {field_id}")
else:
    # Fall back: try to enumerate record sets via records()
    print("No .record_sets property found. Listing top-level record sets by attempting records()...\n")
    # Since we need record set ids, probe dataset._dataset['recordSet'] if present
    recsets = []
    if hasattr(dataset, '_dataset') and 'recordSet' in dataset._dataset:
        record_sets = dataset._dataset['recordSet']
        if isinstance(record_sets, dict):
            record_sets = [record_sets]
        recsets = record_sets
    for record_set in recsets:
        print(f"- {record_set['@id']} (name: {record_set.get('name', '-')})")
        if 'field' in record_set:
            fields = record_set['field']
            if not isinstance(fields, list):
                fields = [fields]
            for field in fields:
                field_id = field['@id'] if isinstance(field, dict) and '@id' in field else str(field)
                print(f"    - field @id: {field_id}")

## 3. Data Extraction
Load data from specific record set(s) into pandas DataFrames. All references use the respective `@id` values.

In [ ]:
# Identify all record set @id's from the metadata
record_set_objs = []
if hasattr(dataset, 'record_sets'):
    record_set_objs = dataset.record_sets
elif hasattr(dataset, '_dataset') and 'recordSet' in dataset._dataset:
    record_sets = dataset._dataset['recordSet']
    if isinstance(record_sets, dict):
        record_sets = [record_sets]
    record_set_objs = record_sets
else:
    raise ValueError('No record sets found in dataset metadata.')

record_set_ids = [rs['@id'] for rs in record_set_objs]
print("Available record set @id's:")
for rsid in record_set_ids:
    print(f"- {rsid}")

# Load each record set entirely into a dictionary of dataframes
dataframes = {}
for rsid in record_set_ids:
    records_iter = dataset.records(record_set=rsid)
    records = list(records_iter)
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"Loaded {len(df)} records for record set {rsid}.")
    else:
        print(f"No records found for record set {rsid}.")

# For demonstration, pick the first loaded record set
if len(dataframes) == 0:
    raise ValueError('No dataframes were loaded from any record set!')
record_set_id = list(dataframes.keys())[0]
print(f"\nColumns in record set {record_set_id}:")
print(dataframes[record_set_id].columns.tolist())
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping. All fields referenced by their `@id` values.

In [ ]:
# --- EDA ---
# Choose a numeric field @id and a grouping field @id for this section

# List all columns (which are field @id's)
df = dataframes[record_set_id]
print("Record set columns / field @id's:")
print(list(df.columns))

# Attempt to select a numeric field
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field is None:
    # Try to coerce any column
    for col in df.columns:
        try:
            pd.to_numeric(df[col].dropna().iloc[:10])
            numeric_field = col
            df[numeric_field] = pd.to_numeric(df[col], errors='coerce')
            break
        except:
            continue
if numeric_field is None:
    raise Exception("No numeric field found in record set.")

print(f"\nUsing numeric field: {numeric_field}")

# Filter: keep rows where numeric_field > threshold
threshold = df[numeric_field].dropna().mean()  # Use mean as threshold for demonstration
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
print(filtered_df[[numeric_field]].head())

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field (choose the first string/object column different from the numeric field)
group_field = None
for col in df.columns:
    if col == numeric_field:
        continue
    if pd.api.types.is_object_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col]):
        group_field = col
        break

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())
else:
    print("No suitable group field detected for grouping.")

## 5. Visualization
Visualize data distributions and relationships using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If grouping field was found, boxplot
if group_field and group_field in df.columns:
    plt.figure(figsize=(12, 5))
    order = df[group_field].value_counts().index[:6]  # Up to 6 most frequent groups
    sns.boxplot(
        data=df[df[group_field].isin(order)],
        x=group_field,
        y=numeric_field,
        order=order
    )
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
Using the `mlcroissant` library, we've loaded FAIR^2 dataset metadata and records, identified record sets and field structures by their `@id` values, and performed initial EDA and visualization. This workflow demonstrates how Croissant enables transparent, reproducible access to structured tabular data for deeper statistical and ML analysis.